# Lesson 14 — Computational psychiatry: linking model parameters to psychopathology

Everything so far has been about describing *how* a person learns and decides: a handful of numbers — a learning rate, an inverse temperature, a weight on model-based values, a volatility learning rate — that summarise many trials of behaviour. **Computational psychiatry** asks whether those numbers differ with symptoms of mental illness, and whether they can tell us something about mechanisms that a questionnaire or a diagnosis cannot (Montague et al., 2012; Huys, Maia & Frank, 2016). A fitted parameter used this way is a *computational phenotype* (Patzelt, Hartley & Gershman, 2018).

The idea is powerful and the pitfalls are many. This lesson walks through three analyses that mirror influential studies, then takes the pitfalls one by one: identifiability, reliability, confounds, power, and what a correlation can and cannot mean.

**A note on the data.** The three data sets used here are **simulated** by the models of this course (lesson 9, 12 and 13 models) with symptom scores that were *constructed* to relate to specific parameters, *inspired by* the published findings cited below. They are not real participants, and nothing here is evidence about any clinical population. The advantage is that the ground truth is known, so at the end of each analysis we can ask how close we got.

**What you will learn**

- how to turn a fitted parameter (or a model-agnostic index) into a test of a hypothesis about symptoms, controlling for covariates,
- three worked examples: compulsivity and model-based control (Gillan et al., 2016), anxiety and learning-rate adaptation to volatility (Browning et al., 2015), anhedonia and learning from positive feedback (Huys et al., 2013; Lefebvre et al., 2017),
- why reliability caps every correlation you can observe, and how to measure and improve it (Brown et al., 2020; Zorowitz & Niv, 2023),
- power analysis by simulation,
- a checklist for a defensible parameter–symptom analysis.

**Prerequisites:** lessons 5–9, 12 and 13.

**Estimated time:** 2.5 hours.

In [ ]:
import sys; sys.path.insert(0, "..")      # find the rlcourse package when running from notebooks/
import numpy as np
import matplotlib.pyplot as plt
from rlcourse import check, hint, reveal, progress, use, setup_style
from rlcourse import models, tasks, fitting, plotting
setup_style()
rng = np.random.default_rng(2024)        # one seed per notebook: results are reproducible

In [ ]:
import pandas as pd                      # this lesson also needs pandas, scipy.stats, statsmodels and the data sets
import warnings
from scipy import stats
import statsmodels.api as sm
from rlcourse import datasets

RUN_FULL = False     # True: refit every participant live (several minutes); False: load the cached fits
warnings.filterwarnings("ignore", message="overflow encountered in exp")   # runaway beta estimates overflow exp(); harmless

def correlation_with_ci(x, y):
    """Pearson r with a 95% confidence interval (Fisher z transform) and p-value."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    r, p = stats.pearsonr(x, y)
    z, se = np.arctanh(r), 1 / np.sqrt(len(x) - 3)
    return r, p, np.tanh(z - 1.96 * se), np.tanh(z + 1.96 * se)

## 14.1 Parameters as phenotypes

A questionnaire score summarises what people *say* about themselves; a model parameter summarises what they *do* in a controlled task, through the lens of a theory of how the task is solved. If a symptom dimension goes with, say, a smaller weight on model-based values, that is a statement about a mechanism — the balance between planning and habit — that no questionnaire item measures directly. The logic of a parameter–symptom study is therefore:

1. choose a task in which the mechanism of interest shows in behaviour,
2. fit a model whose parameter indexes that mechanism (and validate the model: lessons 6–7),
3. relate the parameter to symptoms across people, controlling for things that could drive both,
4. show that the parameter is *recoverable* and *reliable* enough for individual differences,
5. interpret cautiously: the data are correlational.

Two flavours of this programme exist. **Theory-driven** work (this lesson) starts from a model and asks how its parameters vary. **Data-driven** work starts from large behavioural data sets and looks for structure (Huys et al., 2016). Both need the same methodological care.

The three data sets each come with a *trials* table, a *subjects* table with questionnaire scores, and a hidden *truth* table with the generating parameters:

In [ ]:
for name in ["two_step_compulsivity", "reversal_anxiety", "bandit_anhedonia"]:
    print(f"{name}:\n   {datasets.describe(name)}\n")

## 14.2 Worked example 1: compulsivity and model-based control

Gillan, Kosinski, Whelan, Phelps & Daw (2016) had nearly 2,000 online participants play the two-step task (lesson 9) and fill in questionnaires. A transdiagnostic *compulsive behaviour and intrusive thought* dimension was associated with reduced model-based control, over and above diagnoses. Our simulated analogue has 60 participants, 200 trials each, a `compulsivity` score built to relate to the model-based weight $w$, and an `anxiety` score built to relate to nothing.

In [ ]:
trials_ts, subjects_ts = datasets.load("two_step_compulsivity")
per_subject_ts = datasets.split_by_subject(trials_ts)        # {subject: {"a1": ..., "s2": ..., "a2": ..., "r": ..., "common": ...}}
print(trials_ts.head())
print(subjects_ts.describe().round(1))

### A model-agnostic first look

Before any fitting, look at the behavioural signature itself. In lesson 9 you learned that model-free control shows as a main effect of reward on staying, and model-based control as a reward × transition interaction. Let's compare the stay-probability pattern of the most and least compulsive thirds of the sample.

In [ ]:
subjects_ts["comp_tertile"] = pd.qcut(subjects_ts["compulsivity"].rank(method="first"), 3, labels=["low", "medium", "high"])
stay_tables = {s: models.stay_probabilities(d) for s, d in per_subject_ts.items()}

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), sharey=True)
for ax, tertile in zip(axes, ["low", "high"]):
    ids = subjects_ts.loc[subjects_ts["comp_tertile"] == tertile, "subject"]
    mean_table = np.mean([stay_tables[s] for s in ids], axis=0)
    plotting.plot_stay_probabilities(mean_table, ax=ax, title=f"{tertile} compulsivity (n = {len(ids)})", ylim=(0.4, 1.0))
plt.tight_layout()
plt.show()

The interaction (the crossing of the bars) is the model-based signature; it is visibly weaker in the high-compulsivity third. To quantify it per person, the field uses a **one-trial-back logistic regression**: the probability of repeating the stage-1 choice is regressed on the previous trial's reward (±1), its transition type (±1) and their product. The reward coefficient is the model-free index and the interaction coefficient the model-based index (Daw et al., 2011; Gillan et al., 2016).

### ✏️ Exercise 14.1 — The one-trial-back regression *(coding)*

Implement the regression for one participant. Input: a dict with arrays `a1` (stage-1 choices), `r` (rewards), `common` (1 if the transition was common). Output: a tuple `(intercept, b_reward, b_transition, b_interaction)` from `statsmodels.api.Logit`. Code reward and transition as +1/−1 and remember that staying on trial $t$ depends on what happened on trial $t-1$.

In [ ]:
def stay_regression_coefficients(data):
    """Logistic regression of staying (stage-1 choice repeats) on the previous trial's
    reward (+1/-1), transition (+1 common / -1 rare) and their interaction.
    Returns (intercept, b_reward, b_transition, b_interaction)."""
    a1 = np.asarray(data["a1"])
    r = np.asarray(data["r"], dtype=float)
    common = np.asarray(data["common"], dtype=float)
    stay = ...              # <- 1 if the choice on trial t equals the choice on trial t-1 (length n-1)
    reward = ...            # <- previous trial's reward coded +1 / -1 (length n-1)
    transition = ...        # <- previous trial's transition coded +1 / -1 (length n-1)
    X = np.column_stack([np.ones(len(stay)), reward, transition, reward * transition])
    result = sm.Logit(stay, X).fit(disp=0)
    return tuple(float(b) for b in result.params)

check("14.1", stay_regression_coefficients)

In [ ]:
stay_regression_coefficients = use("14.1", stay_regression_coefficients)

rows = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore")           # a few participants produce convergence warnings
    for s, d in per_subject_ts.items():
        try:
            b0, b_rew, b_trans, b_int = stay_regression_coefficients(d)
        except Exception:                     # perfect separation etc.: mark as missing
            b0 = b_rew = b_trans = b_int = np.nan
        rows.append({"subject": s, "mf_index": b_rew, "mb_index": b_int})
indices_ts = pd.DataFrame(rows).merge(subjects_ts, on="subject")
print(indices_ts[["mf_index", "mb_index"]].describe().round(2))

# Logistic-regression coefficients explode for participants who (almost) never switch after a reward
# (quasi-separation): a few |coefficients| > 3 would dominate every correlation. We winsorise the
# indices at their 5th and 95th percentiles — a standard, transparent fix (report it!).
for col in ["mf_index", "mb_index"]:
    lo_q, hi_q = indices_ts[col].quantile([0.05, 0.95])
    indices_ts[col] = indices_ts[col].clip(lo_q, hi_q)
print("after winsorising:")
print(indices_ts[["mf_index", "mb_index"]].describe().round(2))

### ✏️ Exercise 14.2 — A model-based index from a stay-probability table *(maths)*

A participant's stay probabilities are: rewarded–common 0.80, rewarded–rare 0.64, unrewarded–common 0.55, unrewarded–rare 0.72. Using the definition

$$\text{MB index} = \big[P(\text{stay}\mid \text{rew, common}) - P(\text{stay}\mid \text{rew, rare})\big] - \big[P(\text{stay}\mid \text{unrew, common}) - P(\text{stay}\mid \text{unrew, rare})\big],$$

what is this participant's model-based index?

In [ ]:
answer = None   # <- your number
check("14.2", answer)

### The model-based parameter $w$

The regression index is quick and robust, but it is a proxy. The parameter that *means* "weight on model-based values" is $w$ of the hybrid model (lesson 9). Fitting seven parameters to 60 participants takes a few minutes, so the course ships cached maximum-likelihood fits (`data/two_step_compulsivity_fits.csv`, produced by `scripts/make_fits.py` with 5 random starts per participant). The cell below fits three participants live so that you see the procedure, then loads the rest.

In [ ]:
def fit_two_step_subject(data, n_starts, rng):
    fit = fitting.fit_mle(models.two_step_nll, (data,), models.TWO_STEP_PARAM_KINDS, n_starts=n_starts, rng=rng)
    return dict(zip(models.TWO_STEP_PARAM_NAMES, fit["params"]), nll=fit["nll"])

if RUN_FULL:
    fits_ts = pd.DataFrame([dict(subject=s, **fit_two_step_subject(d, 5, np.random.default_rng(1000 + s)))
                            for s, d in per_subject_ts.items()])
else:
    fits_ts = pd.read_csv("../data/two_step_compulsivity_fits.csv")
    for s in [0, 1, 2]:                                       # three live fits, for illustration
        live = fit_two_step_subject(per_subject_ts[s], 2, np.random.default_rng(1000 + s))
        print(f"subject {s}: live fit w = {live['w']:.2f} (nll {live['nll']:.1f});  cached w = {fits_ts.loc[s, 'w']:.2f} (nll {fits_ts.loc[s, 'nll']:.1f})")
fits_ts = fits_ts.merge(indices_ts, on="subject")
print(fits_ts[models.TWO_STEP_PARAM_NAMES].describe().round(2))

Two things to notice in the table. The learning rates and $w$ have sensible distributions, but the inverse temperatures have a few enormous values: when a participant's stage-2 choices are almost perfectly consistent with the last reward, the likelihood keeps increasing as $\beta_2 \to \infty$ — a **boundary estimate** (lesson 5). In a real analysis you would use priors (MAP) or a hierarchical model (lesson 8) to tame these; for $w$, our parameter of interest, the fits are usable as they are.

How does the fitted $w$ compare with the regression index? Gillan et al. (2016) report a correlation of about 0.87 between the two in their data.

In [ ]:
r, p, lo, hi = correlation_with_ci(fits_ts["w"], fits_ts["mb_index"])
fig, ax = plt.subplots(figsize=(4.8, 3.8))
ax.scatter(fits_ts["mb_index"], fits_ts["w"], s=22, alpha=0.8, edgecolor="white")
ax.set(xlabel="model-based index (regression interaction)", ylabel="fitted w", title=f"Two ways to measure model-based control: r = {r:.2f}")
plt.show()

### Relating $w$ to compulsivity

Now the actual question. We report a correlation with a confidence interval (and a rank correlation, which is robust to outliers), and then a regression that controls for age and anxiety.

In [ ]:
r, p, lo, hi = correlation_with_ci(fits_ts["w"], fits_ts["compulsivity"])
rho, p_rho = stats.spearmanr(fits_ts["w"], fits_ts["compulsivity"])
print(f"Pearson r = {r:.2f} [95% CI {lo:.2f}, {hi:.2f}], p = {p:.4f};   Spearman rho = {rho:.2f}, p = {p_rho:.4f}")

fig, ax = plt.subplots(figsize=(5, 3.8))
ax.scatter(fits_ts["compulsivity"], fits_ts["w"], s=24, alpha=0.8, edgecolor="white")
slope, intercept = np.polyfit(fits_ts["compulsivity"], fits_ts["w"], 1)
xx = np.linspace(fits_ts["compulsivity"].min(), fits_ts["compulsivity"].max(), 50)
ax.plot(xx, intercept + slope * xx, color="#52514e", lw=1.5)
ax.set(xlabel="compulsivity score", ylabel="fitted model-based weight w", title="Compulsivity and model-based control (simulated data)")
plt.show()

### ✏️ Exercise 14.3 — Controlling for covariates *(coding)*

Write a function that regresses a fitted parameter on a symptom score with optional covariates, with every variable **z-scored** so that the coefficient is a standardised beta. Return `(beta_symptom, p_value)` from `statsmodels.api.OLS`.

In [ ]:
def symptom_regression(df, param, symptom, covariates=()):
    """Standardised OLS regression of df[param] on df[symptom] (+ covariates).
    Returns (standardised beta of the symptom, its p-value)."""
    cols = [param, symptom] + list(covariates)
    z = ...                                          # <- z-score every column in cols
    X = sm.add_constant(z[[symptom] + list(covariates)].to_numpy())
    result = ...                                     # <- fit the OLS model of z[param] on X
    return float(result.params[1]), float(result.pvalues[1])

check("14.3", symptom_regression)

In [ ]:
symptom_regression = use("14.3", symptom_regression)

table = []
for param in ["w", "mb_index", "mf_index", "alpha2", "lam"]:
    for symptom in ["compulsivity", "anxiety"]:
        beta_std, p = symptom_regression(fits_ts.dropna(), param, symptom, covariates=("age",))
        table.append({"parameter": param, "symptom": symptom, "std. beta (age-controlled)": round(beta_std, 2), "p": round(p, 4)})
pd.DataFrame(table).pivot(index="parameter", columns="symptom")

The relation is **specific**: compulsivity relates to the model-based measures and to nothing else, and anxiety relates to nothing — which is what was built into the data. Specificity matters in real studies too: if a symptom score correlated with *every* parameter, the likely explanation would be a general factor (attention, effort, understanding of the instructions) rather than a mechanism.

### ✏️ Exercise 14.4 — What a parameter–symptom correlation shows *(conceptual)*

Fitted $w$ correlates negatively with compulsivity ($r = -0.3$, $p < .01$) in a large online sample. Which conclusion is justified?

**(a)** People with compulsive symptoms cannot plan.
**(b)** In this task, higher compulsivity goes with a lower estimated weight on model-based values — provided $w$ is recoverable and reliable and the relation survives obvious confounds.
**(c)** Compulsivity causes a reduction in model-based control.
**(d)** The two-step task can be used to diagnose obsessive–compulsive disorder.

In [ ]:
check("14.4", None)   # <- replace None with "a", "b", "c" or "d"

### The advantage of simulated data: how close did we get?

Because the data are simulated we can compare the fitted $w$ with the generating value — a parameter-recovery check (lesson 6) in *exactly* this design — and the observed correlation with the one that was built in.

In [ ]:
truth_ts = datasets.truth("two_step_compulsivity")
r_rec = np.corrcoef(truth_ts["w"], fits_ts.sort_values("subject")["w"])[0, 1]
r_true, *_ = correlation_with_ci(truth_ts["w"], subjects_ts["compulsivity"])
r_obs, *_ = correlation_with_ci(fits_ts["w"], fits_ts["compulsivity"])
print(f"recovery of w in this design: r(true, fitted) = {r_rec:.2f}")
print(f"correlation with compulsivity: true w {r_true:.2f}  vs  fitted w {r_obs:.2f}")

fig, ax = plt.subplots(figsize=(4.4, 3.8))
ax.scatter(truth_ts["w"], fits_ts.sort_values("subject")["w"], s=22, alpha=0.8, edgecolor="white")
ax.plot([0, 1], [0, 1], ls="--", color="#52514e", lw=1)
ax.set(xlabel="true w", ylabel="fitted w (MLE, 200 trials)", title=f"Parameter recovery of w: r = {r_rec:.2f}")
plt.show()

The observed correlation is clearly weaker than the true one. Nothing went wrong: estimation noise in $w$ **attenuates** every correlation it enters, a theme we return to in section 14.5. With 200 trials and maximum likelihood, $w$ is only moderately recoverable; hierarchical estimation (lesson 8) and more trials both help.

## 14.3 Worked example 2: anxiety and learning-rate adaptation to volatility

Browning, Behrens, Jocham, O'Reilly & Bishop (2015) had participants learn in a stable and a volatile block (the design of Behrens et al., 2007; lesson 12). Healthy learners raise their learning rate in the volatile block; people high in trait anxiety did so less — their learning rate was less *adapted* to the statistics of the environment (see also Gagne et al., 2020; Pulcu & Browning, 2019). The simulated `reversal_anxiety` data set builds the same relation into a Rescorla–Wagner learner with one learning rate per block.

The analysis is simple: fit the RW model (two parameters: $\alpha$, $\beta$) separately to each participant's stable and volatile block, and compute the adaptation $\Delta\alpha = \text{logit}(\alpha_{\text{volatile}}) - \text{logit}(\alpha_{\text{stable}})$. Two parameters and 120 trials per block: fast enough to fit everyone live.

In [ ]:
trials_rv, subjects_rv = datasets.load("reversal_anxiety")
per_subject_rv = datasets.split_by_subject(trials_rv)

def fit_rw_block(outcomes, choices, rng):
    fit = fitting.fit_mle(models.rw_reversal_nll, (outcomes, choices), ["unit", "pos"], n_starts=3, rng=rng)
    return fit["params"]           # (alpha, beta)

rows = []
for s, d in per_subject_rv.items():
    out = {"subject": s}
    for block in ["stable", "volatile"]:
        mask = d["block"] == block
        alpha, beta = fit_rw_block(d["outcome"][mask], d["choice"][mask], np.random.default_rng(3000 + s))
        out[f"alpha_{block}"], out[f"beta_{block}"] = alpha, beta
    rows.append(out)
fits_rv = pd.DataFrame(rows).merge(subjects_rv, on="subject")
fits_rv["adaptation"] = models.logit(fits_rv["alpha_volatile"]) - models.logit(fits_rv["alpha_stable"])
print(fits_rv[["alpha_stable", "alpha_volatile", "adaptation"]].describe().round(2))

In [ ]:
r, p, lo, hi = correlation_with_ci(fits_rv["adaptation"], fits_rv["trait_anxiety"])
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.8))
axes[0].scatter(fits_rv["alpha_stable"], fits_rv["alpha_volatile"], s=22, alpha=0.8, edgecolor="white")
axes[0].plot([0, 1], [0, 1], ls="--", color="#52514e", lw=1)
axes[0].set(xlabel="α in the stable block", ylabel="α in the volatile block", title="Most people raise their learning rate\nwhen the world becomes volatile")
axes[1].scatter(fits_rv["trait_anxiety"], fits_rv["adaptation"], s=22, alpha=0.8, edgecolor="white")
axes[1].axhline(0, color="#52514e", lw=0.8, ls=":")
axes[1].set(xlabel="trait anxiety", ylabel="adaptation  logit(α_vol) − logit(α_stable)", title=f"Anxiety and learning-rate adaptation: r = {r:.2f}\n95% CI [{lo:.2f}, {hi:.2f}]")
plt.tight_layout()
plt.show()

beta_std, p = symptom_regression(fits_rv, "adaptation", "trait_anxiety", covariates=("age", "compulsivity"))
print(f"standardised beta of trait anxiety (controlling for age and compulsivity) = {beta_std:.2f}, p = {p:.4f}")

Two remarks. First, the *difference* between two noisy estimates is noisier than either, so adaptation scores are harder to measure reliably than a single learning rate — one reason to prefer a model with an explicit adaptation parameter, such as the VKF's volatility learning rate $\lambda$ (lesson 12), fitted to the whole session. Second, the interpretation offered by Browning et al. and by Pulcu & Browning (2019) is not "anxious people learn too slowly" but "their learning rate is less *tuned* to the environment": a dysregulation rather than a deficit — a distinction that Sharp, Dolan & Eldar (2023) also draw for state-transition learning and compulsivity.

## 14.4 Worked example 3: anhedonia and learning from positive feedback

Anhedonia — reduced pleasure and motivation — is the symptom most often linked to reward learning. The meta-analysis of Huys, Pizzagalli, Bogdan & Dayan (2013) found that anhedonia related most consistently to a reduced *sensitivity* to reward (how much a reward is worth, which scales values) rather than to the learning rate as such. Our simulated `bandit_anhedonia` data set instead builds the relation into the **positive learning rate** $\alpha^+$ of the asymmetric model of lesson 13 (Lefebvre et al., 2017), for didactic reasons: it lets us illustrate a parameter trade-off below. The data are a two-armed bandit with one reversal (160 trials), and the cached fits contain all five candidate models of the course (lesson 15 compares them).

In [ ]:
trials_bd, subjects_bd = datasets.load("bandit_anhedonia")
fits_bd_all = pd.read_csv("../data/bandit_anhedonia_fits.csv")
fits_bd = fits_bd_all[fits_bd_all["model"] == "rw_asym"].merge(subjects_bd, on="subject")
print(fits_bd[["alpha_pos", "alpha_neg", "beta"]].describe().round(2))

table = []
for param in ["alpha_pos", "alpha_neg", "beta"]:
    for symptom in ["anhedonia", "anxiety"]:
        b, p = symptom_regression(fits_bd, param, symptom, covariates=("age",))
        table.append({"parameter": param, "symptom": symptom, "std. beta": round(b, 2), "p": round(p, 4)})
pd.DataFrame(table).pivot(index="parameter", columns="symptom")

The $\beta$ column deserves a look: the maximum is absurd. A handful of participants have near-deterministic choices and their $\beta$ estimate ran away. Whenever a parameter's distribution looks like this, use a rank correlation, analyse $\log\beta$, or cap the estimate — and consider priors.

The more important check is the **trade-off** between parameters. Learning rates and the inverse temperature partly compensate for each other: a lower $\alpha^+$ with a higher $\beta$ can produce choices similar to a higher $\alpha^+$ with a lower $\beta$ (Katahira, 2015; Eckstein, Wilbrecht & Collins, 2021). If the fitted $\alpha^+$ and $\beta$ were strongly correlated across people, a correlation between $\alpha^+$ and anhedonia could be a correlation between $\beta$ and anhedonia in disguise.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.8))
axes[0].scatter(fits_bd["anhedonia"], fits_bd["alpha_pos"], s=22, alpha=0.8, edgecolor="white")
r, p, lo, hi = correlation_with_ci(fits_bd["alpha_pos"], fits_bd["anhedonia"])
axes[0].set(xlabel="anhedonia score", ylabel="fitted α⁺", title=f"Anhedonia and the positive learning rate: r = {r:.2f}")
rho = stats.spearmanr(fits_bd["alpha_pos"], fits_bd["beta"])[0]
axes[1].scatter(fits_bd["alpha_pos"], np.log(fits_bd["beta"]), s=22, alpha=0.8, edgecolor="white")
axes[1].set(xlabel="fitted α⁺", ylabel="log fitted β", title=f"Trade-off check: Spearman ρ(α⁺, β) = {rho:.2f}")
plt.tight_layout()
plt.show()

The trade-off is substantial: across participants the fitted $\alpha^+$ and $\beta$ correlate at about $\rho \approx -0.5$ — and because the data are simulated we know that the *generating* $\alpha^+$ and $\beta$ were drawn independently. The correlation is entirely an estimation artefact: the likelihood cannot fully separate "learns fast about rewards" from "chooses deterministically". What protects the conclusion is the specificity table above: $\beta$ itself is unrelated to anhedonia, so the $\alpha^+$ relation is not $\beta$ in disguise. In general the remedies are a task design that de-correlates the parameters (more trials, more varied outcomes), hierarchical estimation, and always reporting the symptom relation for *all* parameters, not just the favourite one.

## 14.5 Reliability: the ceiling on every correlation

A fitted parameter is a measurement, and measurements have **reliability**: the proportion of their variance across people that is stable signal rather than noise. Classical test theory gives the sobering formula

$$ r_{\text{observed}} = r_{\text{true}} \sqrt{\text{rel}_x \, \text{rel}_y} , $$

so two measures with reliabilities 0.5 and 0.8 can never correlate above $\sqrt{0.4} \approx 0.63$, however strong the true relation. Hedge, Powell & Sumner (2018) called it the *reliability paradox*: tasks with robust group effects often have poor reliability precisely because people do not differ much on them. For model parameters, Brown, Chen, Gillan & Price (2020) showed that the reliability of model-based indices depends strongly on the estimation method (hierarchical fitting helped), and Zorowitz & Niv (2023) review design and analysis strategies to improve it.

The simplest estimate is **split-half reliability**: fit the parameter separately to two halves of each participant's data and correlate the two estimates across people. Because each half has only half the trials, the correlation is corrected to full length with the **Spearman–Brown formula** $r_{\text{full}} = 2r / (1 + r)$. (Test–retest reliability across sessions is better still; `fitting.icc_oneway` computes the intraclass correlation for that case.)

### ✏️ Exercise 14.5 — Split-half reliability *(coding)*

Given two arrays of per-participant estimates (one per half), return the Spearman–Brown-corrected split-half reliability.

In [ ]:
def split_half_reliability(x_half1, x_half2):
    """Spearman–Brown corrected correlation between two half-session estimates."""
    r_half = ...          # <- Pearson correlation between the halves
    return ...            # <- correct it to full length: 2r / (1 + r)

check("14.5", split_half_reliability)

### ✏️ Exercise 14.6 — Spearman–Brown by hand *(maths)*

The estimates from the two halves correlate at $r = 0.5$. What is the Spearman–Brown-corrected reliability of the full session?

In [ ]:
answer = None   # <- your number
check("14.6", answer)

### ✏️ Exercise 14.7 — The ceiling on an observable correlation *(maths)*

A parameter has reliability 0.5 and a questionnaire has reliability 0.8. What is the largest correlation between them you can expect to observe, even if the true relation were perfect?

In [ ]:
answer = None   # <- your number
check("14.7", answer)

Let's measure the split-half reliability of $\alpha^+$, $\alpha^-$ and $\beta$ in the anhedonia data in two ways, because — as Brown et al. (2020) showed — the *analysis* changes the answer:

- **A. The naive way:** fit the model from scratch to the first 80 and the last 80 trials of each participant. Two problems: the halves are not parallel (the reward probabilities reverse at trial 80), and 80 trials give wild maximum-likelihood estimates.
- **B. The better way:** split the trials into *interleaved* blocks of 10 (odd blocks vs even blocks), replay the whole sequence in each fit but score only the block's trials (the cross-validation trick of lesson 7), and regularise the estimates with a weak prior (MAP estimation, lesson 8). Both halves then cover both reward regimes, the model's state is carried over, and runaway estimates are tamed.

In [ ]:
split_half_reliability = use("14.5", split_half_reliability)
per_subject_bd = datasets.split_by_subject(trials_bd)
kinds_asym = ["unit", "unit", "pos"]

def nll_rw_asym_masked(params, choices, rewards, mask, q0=0.5):
    """Asymmetric-RW likelihood counting only the trials where mask is True, while the
    values are updated on every trial (the full history is replayed)."""
    alpha_pos, alpha_neg, beta = params
    Q = np.full(2, q0)
    nll = 0.0
    for t, (c, r) in enumerate(zip(choices, rewards)):
        p = models.softmax(Q, beta)
        if mask[t]:
            nll -= np.log(p[c])
        delta = r - Q[c]
        Q[c] = Q[c] + (alpha_pos if delta > 0 else alpha_neg) * delta
    return nll

weak_prior = fitting.normal_log_prior(mu=[0.0, 0.0, 1.0], sd=[1.5, 1.5, 1.0])    # on logit α⁺, logit α⁻, log β
block = np.arange(160) // 10
estimates = {"A: halves, MLE": [[], []], "B: interleaved blocks, MAP": [[], []]}
for s, d in per_subject_bd.items():
    choices, rewards = d["choice"], d["reward"].astype(float)
    for h, sl in enumerate([slice(0, 80), slice(80, 160)]):                             # A
        fit = fitting.fit_mle(models.nll_rw_asym, (choices[sl], rewards[sl]), kinds_asym, n_starts=2,
                              rng=np.random.default_rng(4000 + s + 100000 * h))
        estimates["A: halves, MLE"][h].append(fit["params"])
    for h, mask in enumerate([block % 2 == 0, block % 2 == 1]):                          # B
        fit = fitting.fit_mle(nll_rw_asym_masked, (choices, rewards, mask), kinds_asym, n_starts=2,
                              rng=np.random.default_rng(4500 + s + 100000 * h), log_prior=weak_prior)
        estimates["B: interleaved blocks, MAP"][h].append(fit["params"])

reliability = {}
for method, (e1, e2) in estimates.items():
    e1, e2 = np.array(e1), np.array(e2)
    reliability[method] = {"α⁺": split_half_reliability(e1[:, 0], e2[:, 0]),
                           "α⁻": split_half_reliability(e1[:, 1], e2[:, 1]),
                           "log β": split_half_reliability(np.log(e1[:, 2]), np.log(e2[:, 2]))}
display(pd.DataFrame(reliability).round(2))

fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
for ax, method in zip(axes, estimates):
    e1, e2 = np.array(estimates[method][0]), np.array(estimates[method][1])
    ax.scatter(e1[:, 0], e2[:, 0], s=20, alpha=0.8, edgecolor="white")
    ax.plot([0, 1], [0, 1], ls="--", color="#52514e", lw=1)
    ax.set(xlabel="α⁺ from half 1", ylabel="α⁺ from half 2", title=f"{method}\nsplit-half reliability of α⁺ = {reliability[method]['α⁺']:.2f}")
plt.tight_layout()
plt.show()

Same data, same model, different analysis: the naive estimates are poor for all three parameters, whereas interleaved blocks with a weak prior nearly double the reliability of $\alpha^+$ and make $\beta$ usable. This is the practical content of Brown et al. (2020) and Zorowitz & Niv (2023): reliability is a property of the whole pipeline, and hierarchical or regularised estimation is often the cheapest improvement available. Even the better numbers are moderate — in a clean simulation without lapses of attention, strategy changes or questionnaire noise — so any real parameter–symptom correlation has to be read against this ceiling.

### ✏️ Exercise 14.8 — Reading a reliability estimate *(conceptual)*

The split-half reliability of fitted $w$ in a design is 0.4 (and the symptom questionnaire's reliability is about 0.9). Which statement follows?

**(a)** Even a perfect true relation between $w$ and the symptom would be expected to appear as $r \approx 0.6$ at most; small observed correlations are not surprising.
**(b)** $w$ is useless as an individual-differences measure and should be abandoned.
**(c)** The true correlation between $w$ and the symptom must be smaller than 0.4.
**(d)** Nothing can raise the reliability: it is a fixed property of the parameter.

In [ ]:
check("14.8", None)   # <- replace None with "a", "b", "c" or "d"

## 14.6 Power by simulation

How many participants does a parameter–symptom study need? For a plain correlation there are formulas, but the honest answer for a *model-based* analysis comes from simulation: generate participants from the model with the hypothesised relation built in, fit them, run the planned test, and count how often it is significant. That includes the estimation noise automatically. The exercise implements the core step for a correlation; the "full pipeline" version just replaces the bivariate normal draw with "simulate and fit".

### ✏️ Exercise 14.9 — Power by simulation *(coding)*

Estimate the power of a Pearson correlation test: draw `n_sims` samples of `n` bivariate-normal pairs with correlation `r_true`, test each with `scipy.stats.pearsonr`, and return the proportion of samples with $p <$ `alpha`.

In [ ]:
def power_by_simulation(r_true, n, n_sims, rng, alpha=0.05):
    """Proportion of simulated samples (n pairs with correlation r_true) in which pearsonr gives p < alpha."""
    cov = np.array([[1.0, r_true], [r_true, 1.0]])
    n_significant = 0
    for _ in range(n_sims):
        xy = rng.multivariate_normal([0.0, 0.0], cov, size=n)
        raise NotImplementedError("test the sample and count it if significant")
    return n_significant / n_sims

check("14.9", power_by_simulation)

In [ ]:
power_by_simulation = use("14.9", power_by_simulation)

ns = [20, 40, 60, 100, 150, 200]
fig, ax = plt.subplots(figsize=(6.5, 3.8))
for r_true in [0.2, 0.3, 0.4]:
    power = [power_by_simulation(r_true, n, 200, rng) for n in ns]
    ax.plot(ns, power, "o-", label=f"true r = {r_true}")
ax.axhline(0.8, color="#52514e", ls="--", lw=1, label="80% power")
ax.set(xlabel="number of participants", ylabel="power (α = 0.05)", title="Power of a correlation test, by simulation", ylim=(0, 1.02))
ax.legend()
plt.show()

A correlation of 0.3 — a typical *observed* parameter–symptom effect after attenuation — needs on the order of 80–90 participants for 80% power, and 0.2 needs about 200. Studies with 30 participants per group are underpowered for anything but large effects, which is one reason the field moved to large online samples (Gillan et al., 2016).

## 14.7 Pitfalls and better practice

**Identifiability and trade-offs.** A parameter is interpretable only if the design can recover it (lesson 6) and it does not trade off with another (section 14.4). Eckstein, Wilbrecht & Collins (2021) show that the "same" learning rate can mean different things in different tasks and age groups, because the model absorbs whatever process produces the behaviour. Always report recovery in *your* design.

**Point estimates ignore uncertainty.** Correlating MLE point estimates with symptoms treats a wildly uncertain $\beta = 10^6$ like a precise one. The principled alternative is to put the covariate *inside* the hierarchical model: let each participant's parameter be drawn from a group distribution whose mean depends on the symptom score, and estimate the regression coefficient together with everything else (Piray et al., 2019, do this with HBI; it is natural in Stan or PyMC). The code below does this for the asymmetric model with anhedonia as a predictor of $\text{logit}(\alpha^+)$, for 20 participants. It is optional: it runs automatically when PyMC and the JAX sampler (numpyro) are installed (about 10–20 seconds), and is skipped otherwise.

In [ ]:
import importlib.util, os
RUN_PYMC = all(importlib.util.find_spec(m) is not None for m in ["pymc", "numpyro", "arviz"])   # set False to skip
if RUN_PYMC:
    os.environ.setdefault("PYTENSOR_FLAGS", "cxx=")   # avoid the C compiler; the JAX sampler does the work
    try:
        import pymc as pm, pytensor, pytensor.tensor as pt, arviz as az
    except ImportError as exc:
        print("PyMC is not installed:", exc)
    else:
        sub_ids = sorted(per_subject_bd)[:20]                               # 20 participants keep it quick
        C = np.stack([per_subject_bd[s]["choice"] for s in sub_ids]).astype(int)
        R = np.stack([per_subject_bd[s]["reward"] for s in sub_ids]).astype(float)
        z_anh = ((subjects_bd.set_index("subject").loc[sub_ids, "anhedonia"] - subjects_bd["anhedonia"].mean())
                 / subjects_bd["anhedonia"].std()).to_numpy()
        n_subj, n_trials = C.shape
        idx = pt.arange(n_subj)

        def update(c, r, Q, a_pos, a_neg):                                   # one trial for all participants at once
            q_c = Q[idx, c]
            delta = r - q_c
            alpha = pt.where(delta > 0, a_pos, a_neg)
            return pt.set_subtensor(Q[idx, c], q_c + alpha * delta)

        warnings.simplefilter("ignore")                                     # pytensor's scan API deprecation notice
        with pm.Model() as embedded:
            mu_pos = pm.Normal("mu_pos", 0, 1.5)
            b_anh = pm.Normal("b_anh", 0, 1)                                # <- the covariate effect, estimated jointly
            sd_pos = pm.HalfNormal("sd_pos", 1)
            mu_neg, sd_neg = pm.Normal("mu_neg", 0, 1.5), pm.HalfNormal("sd_neg", 1)
            mu_beta, sd_beta = pm.Normal("mu_beta", 1, 1), pm.HalfNormal("sd_beta", 1)
            x_pos = pm.Normal("x_pos", mu_pos + b_anh * z_anh, sd_pos, shape=n_subj)
            x_neg = pm.Normal("x_neg", mu_neg, sd_neg, shape=n_subj)
            x_beta = pm.Normal("x_beta", mu_beta, sd_beta, shape=n_subj)
            a_pos = pm.Deterministic("alpha_pos", pm.math.sigmoid(x_pos))
            a_neg = pm.Deterministic("alpha_neg", pm.math.sigmoid(x_neg))
            beta = pm.Deterministic("beta", pm.math.exp(x_beta))
            scan_out = pytensor.scan(fn=update, sequences=[pt.as_tensor(C.T), pt.as_tensor(R.T)],
                                     outputs_info=[pt.full((n_subj, 2), 0.5)], non_sequences=[a_pos, a_neg])
            Qs = scan_out[0] if isinstance(scan_out, tuple) else scan_out
            Qs = pt.concatenate([pt.full((1, n_subj, 2), 0.5), Qs[:-1]], axis=0)      # values before each choice
            logp = pm.math.log_softmax(beta[None, :, None] * Qs, axis=2)
            ll = logp[pt.arange(n_trials)[:, None], pt.arange(n_subj)[None, :], pt.as_tensor(C.T)].sum()
            pm.Potential("ll", ll)
            try:
                idata = pm.sample(draws=400, tune=400, chains=2, target_accept=0.9, nuts_sampler="numpyro",
                                  progressbar=False, random_seed=1)
            except Exception:
                idata = pm.sample(draws=400, tune=400, chains=2, target_accept=0.9, progressbar=False, random_seed=1)
        warnings.filterwarnings("default")
        warnings.filterwarnings("ignore", message="overflow encountered in exp")
        print(az.summary(idata, var_names=["mu_pos", "b_anh", "mu_neg", "mu_beta"]))
        print("b_anh is the effect of one SD of anhedonia on logit(alpha+), estimated with its full posterior uncertainty.")
else:
    print("Skipped: PyMC / numpyro / arviz not installed (pip install pymc numpyro arviz to run this cell).")

Read `b_anh` as the regression coefficient of lesson 14.3, but with its uncertainty inherited from every participant's trial-by-trial likelihood rather than from point estimates. (This quick sketch — two short chains, a centred parameterisation — reports a few divergences and R-hat values slightly above 1; a publication-grade version would use a non-centred parameterisation, four chains and more draws, as in lesson 8.)

**Groups versus dimensions.** Comparing a patient group with controls on a parameter is the classic design, but diagnoses are heterogeneous; dimensional, transdiagnostic analyses (Gillan et al., 2016) often have more power and clearer interpretation. If you do compare groups, fit them *in one* hierarchical model with group as a predictor rather than fitting each group separately with its own prior.

**Deficit versus dysregulation.** "Lower learning rate" is only a deficit if a higher one would do better in the task. Sharp, Dolan & Eldar (2023) found that compulsivity was associated with state-transition learning rates that were *poorly adapted* to the environment (too fast in one task, too slow in another) — a statement about regulation that a single-task deficit framing would miss. The same applies to Browning et al.'s anxiety result.

**Time and context.** Parameters fitted from one session on one day are snapshots. Hitchcock, Fried & Frank (2022) argue that computational psychiatry needs repeated measurement over time and across contexts to capture the dynamics that matter clinically.

**Model mis-specification.** If the model is wrong, the parameter means something else. Collins & Frank (2012) showed that part of what looks like reinforcement learning in standard tasks is working memory; a learning-rate–symptom correlation may then be a working-memory–symptom correlation. Model comparison and posterior predictive checks (lesson 7) are not optional.

**Multiple comparisons.** Five parameters × six symptom scales = thirty tests. Pre-register the primary hypothesis, correct the rest, and estimate effect sizes rather than hunting for $p < .05$.

### A checklist for a parameter–symptom analysis

1. The task produces a behavioural signature of the mechanism, verified by simulation (lessons 1, 3, 9).
2. The model is compared with alternatives and passes posterior predictive checks (lesson 7).
3. The parameter is recoverable in this exact design, and its trade-offs with other parameters are known (lesson 6).
4. The parameter's reliability is estimated and reported; correlations are read against the attenuation ceiling.
5. Covariates and specificity: the relation survives age/IQ/other symptom dimensions and does not appear for every parameter.
6. Uncertainty is propagated (hierarchical estimation; covariates embedded where possible).
7. Power was estimated by simulation before data collection; the primary hypothesis was pre-registered.
8. The interpretation is correlational and task-specific; "deficit" claims are made only when a better value exists.

### ✏️ Exercise 14.10 — Why a significant correlation is not enough *(conceptual, written)*

In 3–5 sentences: a fitted parameter correlates significantly with a symptom score. List the checks you would run before interpreting this as evidence about a mechanism, and say why each matters.

In [ ]:
check("14.10", """
    Write your answer here.
""")

## Summary

- A **computational phenotype** is a fitted parameter used as an individual-differences measure; relating it to symptoms is only as sound as the model (comparison, recovery) and the measurement (reliability) behind it.
- Three analyses mirrored the literature: a lower model-based weight with compulsivity (Gillan et al., 2016), weaker learning-rate adaptation with anxiety (Browning et al., 2015), and a lower positive learning rate with anhedonia (a didactic variant of the reward-learning literature; Huys et al., 2013; Lefebvre et al., 2017).
- Estimation noise **attenuates** correlations: $r_{\text{obs}} = r_{\text{true}}\sqrt{\text{rel}_x\,\text{rel}_y}$. Measure reliability (split-half with the Spearman–Brown correction, or test–retest) and improve it by design and hierarchical estimation.
- Power for the *whole pipeline* comes from simulation.
- Check identifiability and trade-offs, control for covariates and test specificity, propagate uncertainty (embed covariates in the hierarchical model), distinguish deficit from dysregulation, and remember that the data are correlational.

## What's next

Lesson 15 is the capstone: you will run this whole pipeline yourself — descriptive analysis, model comparison, model and parameter recovery, the symptom analysis with reliability and power, a posterior predictive check and a write-up — on the anhedonia data set, and then compare your conclusions with the ground truth.

## References

- Brown, V. M., Chen, J., Gillan, C. M., & Price, R. B. (2020). Improving the reliability of computational analyses: Model-based planning and its relationship with compulsivity. *Biological Psychiatry: Cognitive Neuroscience and Neuroimaging, 5*(6), 601–609. https://doi.org/10.1016/j.bpsc.2019.12.019
- Browning, M., Behrens, T. E., Jocham, G., O'Reilly, J. X., & Bishop, S. J. (2015). Anxious individuals have difficulty learning the causal statistics of aversive environments. *Nature Neuroscience, 18*, 590–596. https://doi.org/10.1038/nn.3961
- Collins, A. G. E., & Frank, M. J. (2012). How much of reinforcement learning is working memory, not reinforcement learning? A behavioral, computational, and neurogenetic analysis. *European Journal of Neuroscience, 35*(7), 1024–1035. https://doi.org/10.1111/j.1460-9568.2011.07980.x
- Daw, N. D., Gershman, S. J., Seymour, B., Dayan, P., & Dolan, R. J. (2011). Model-based influences on humans' choices and striatal prediction errors. *Neuron, 69*(6), 1204–1215. https://doi.org/10.1016/j.neuron.2011.02.027
- Eckstein, M. K., Wilbrecht, L., & Collins, A. G. E. (2021). What do reinforcement learning models measure? Interpreting model parameters in cognition and neuroscience. *Current Opinion in Behavioral Sciences, 41*, 128–137. https://doi.org/10.1016/j.cobeha.2021.06.004
- Gagne, C., Zika, O., Dayan, P., & Bishop, S. J. (2020). Impaired adaptation of learning to contingency volatility in internalizing psychopathology. *eLife, 9*, e61387. https://doi.org/10.7554/eLife.61387
- Gillan, C. M., Kosinski, M., Whelan, R., Phelps, E. A., & Daw, N. D. (2016). Characterizing a psychiatric symptom dimension related to deficits in goal-directed control. *eLife, 5*, e11305. https://doi.org/10.7554/eLife.11305
- Hedge, C., Powell, G., & Sumner, P. (2018). The reliability paradox: Why robust cognitive tasks do not produce reliable individual differences. *Behavior Research Methods, 50*, 1166–1186. https://doi.org/10.3758/s13428-017-0935-1
- Hitchcock, P. F., Fried, E. I., & Frank, M. J. (2022). Computational psychiatry needs time and context. *Annual Review of Psychology, 73*, 243–270. https://doi.org/10.1146/annurev-psych-021621-124910
- Huys, Q. J. M., Maia, T. V., & Frank, M. J. (2016). Computational psychiatry as a bridge from neuroscience to clinical applications. *Nature Neuroscience, 19*, 404–413. https://doi.org/10.1038/nn.4238
- Huys, Q. J. M., Pizzagalli, D. A., Bogdan, R., & Dayan, P. (2013). Mapping anhedonia onto reinforcement learning: A behavioural meta-analysis. *Biology of Mood & Anxiety Disorders, 3*, 12. https://doi.org/10.1186/2045-5380-3-12
- Katahira, K. (2015). The relation between reinforcement learning parameters and the influence of reinforcement history on choice behavior. *Journal of Mathematical Psychology, 66*, 59–69. https://doi.org/10.1016/j.jmp.2015.03.006
- Lefebvre, G., Lebreton, M., Meyniel, F., Bourgeois-Gironde, S., & Palminteri, S. (2017). Behavioural and neural characterization of optimistic reinforcement learning. *Nature Human Behaviour, 1*, 0067. https://doi.org/10.1038/s41562-017-0067
- Mkrtchian, A., Aylward, J., Dayan, P., Roiser, J. P., & Robinson, O. J. (2017). Modeling avoidance in mood and anxiety disorders using reinforcement learning. *Biological Psychiatry, 82*(7), 532–539. https://doi.org/10.1016/j.biopsych.2017.01.017
- Montague, P. R., Dolan, R. J., Friston, K. J., & Dayan, P. (2012). Computational psychiatry. *Trends in Cognitive Sciences, 16*(1), 72–80. https://doi.org/10.1016/j.tics.2011.11.018
- Patzelt, E. H., Hartley, C. A., & Gershman, S. J. (2018). Computational phenotyping: Using models to understand individual differences in personality, development, and mental illness. *Personality Neuroscience, 1*, e18. https://doi.org/10.1017/pen.2018.14
- Piray, P., Dezfouli, A., Heskes, T., Frank, M. J., & Daw, N. D. (2019). Hierarchical Bayesian inference for concurrent model fitting and comparison for group studies. *PLoS Computational Biology, 15*(6), e1007043. https://doi.org/10.1371/journal.pcbi.1007043
- Pulcu, E., & Browning, M. (2019). The misestimation of uncertainty in affective disorders. *Trends in Cognitive Sciences, 23*(10), 865–875. https://doi.org/10.1016/j.tics.2019.07.007
- Sharp, P. B., Russek, E. M., Huys, Q. J. M., Dolan, R. J., & Eldar, E. (2022). Humans perseverate on punishment avoidance goals in multigoal reinforcement learning. *eLife, 11*, e74402. https://doi.org/10.7554/eLife.74402
- Sharp, P. B., Dolan, R. J., & Eldar, E. (2023). Disrupted state transition learning as a computational marker of compulsivity. *Psychological Medicine, 53*(5), 2095–2105. https://doi.org/10.1017/S0033291721003846
- Voon, V., Derbyshire, K., Rück, C., Irvine, M. A., Worbe, Y., Enander, J., … Bullmore, E. T. (2015). Disorders of compulsivity: A common bias towards learning habits. *Molecular Psychiatry, 20*, 345–352. https://doi.org/10.1038/mp.2014.44
- Zorowitz, S., & Niv, Y. (2023). Improving the reliability of cognitive task measures: A narrative review. *Biological Psychiatry: Cognitive Neuroscience and Neuroimaging, 8*(8), 789–797. https://doi.org/10.1016/j.bpsc.2023.02.004